In [ ]:
from huggingface_hub import get_token
assert get_token(), 'Please run hf auth login before executing the notebook.'
print('Hugging Face token is available')


In [ ]:
import torch
from transformers import AutoModelForCausalLM
import matplotlib.pyplot as plt

model = AutoModelForCausalLM.from_pretrained('meta-llama/Llama-3.2-1B',)

In [ ]:
# TODO for Question 4
"""

Hint:
# Analyze the weights of the query projection layer (q_proj) where layer = k
 q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

# Apply SVD to decompose the weight matrix into U (left singular vectors), S (singular values), and V (right singular vectors).
 U, S, Vh = torch.linalg.svd(q_weight.float(), full_matrices=False)

# Compute the retained energy ratio (Cumulative energy) of each singular value.
energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
"""


In [ ]:
import csv
from pathlib import Path

layers_to_analyze = [0, 7, 15]
output_dir = Path('q4_results')
output_dir.mkdir(parents=True, exist_ok=True)

# Use GPU for SVD if Kaggle GPU is enabled. No model fine-tuning is performed.
svd_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('SVD device:', svd_device)

results = []
for k in layers_to_analyze:
    # 1. Get the original q_proj weight matrix.
    q_weight = model.model.layers[k].self_attn.q_proj.weight.detach()

    # 2. Full (not truncated-to-rank) SVD; singular values are descending.
    # Move only this matrix onto the computation device to avoid moving the whole model.
    weight_fp32 = q_weight.to(device=svd_device, dtype=torch.float32)
    U, S, Vh = torch.linalg.svd(weight_fp32, full_matrices=False)

    # 3. Cumulative energy percentage (denominator includes ALL singular values).
    energy_ratios = torch.cumsum(S ** 2, dim=0) / torch.sum(S ** 2) * 100
    min_rank = int(torch.searchsorted(energy_ratios.contiguous(), 50.0).item()) + 1
    results.append({'layer': k, 'minimum_target_rank_50pct': min_rank})
    print(f'Layer {k}: minimum rank for >= 50% energy = {min_rank}')

    # Plot only the first 200 values, as requested in the slides.
    n_plot = min(200, S.numel())
    x = list(range(1, n_plot + 1))
    singular_values = S[:n_plot].cpu().numpy()
    cumulative_energy = energy_ratios[:n_plot].cpu().numpy()

    # 4a. Singular value magnitude: BAR chart.
    plt.figure(figsize=(8, 4.5))
    plt.bar(x, singular_values, width=1.0)
    plt.title(f'Layer {k} q_proj: Singular Value Magnitudes')
    plt.xlabel('Singular Value Index')
    plt.ylabel('Singular Value Magnitude')
    plt.xlim(0, n_plot + 1)
    plt.tight_layout()
    plt.savefig(output_dir / f'layer_{k}_singular_values.png', dpi=200)
    plt.show()
    plt.close()

    # 4b. Cumulative energy: LINE chart.
    plt.figure(figsize=(8, 4.5))
    plt.plot(x, cumulative_energy)
    plt.title(f'Layer {k} q_proj: Cumulative Energy')
    plt.xlabel('Target Rank')
    plt.ylabel('Cumulative Energy (%)')
    plt.xlim(1, n_plot)
    plt.ylim(0, 100)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(output_dir / f'layer_{k}_cumulative_energy.png', dpi=200)
    plt.show()
    plt.close()

    # Release SVD matrices before moving to the next layer.
    del weight_fp32, U, S, Vh, energy_ratios

# 5. Save the 50% energy ranks as a report-ready CSV.
with (output_dir / 'minimum_ranks_50pct.csv').open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['layer', 'minimum_target_rank_50pct'])
    writer.writeheader()
    writer.writerows(results)

print('\nQuestion 4 summary:')
for result in results:
    print(f"Layer {result['layer']}: {result['minimum_target_rank_50pct']}")
print(f'\nSaved 6 plots and CSV to: {output_dir.resolve()}')
